# Pacific Northwest Modeled Streamflow Change
Created for the 30 Day Map Challenge after a recent PNW atmospheric river, this exploratory view shows how NOAA National Water Model (NWM) channel-routing flow changed during Nov 9–14, 2026. Each hexagon represents one selected modeled river reach, not every stream or the land area inside the cell. Values are relative to that reach's actual flow at Nov 9, 00:00 UTC; this is not a historical anomaly, observed-flow map, or measure of flood severity.

<iframe src="map.html" width="100%" height="600" style="border:0; border-radius:8px;" loading="lazy" title="Animated map of Pacific Northwest modeled streamflow change in H3 hexagons"></iframe>

*Press play or drag the slider to step through Nov 9–14, 2026. Height and color show the per-reach flow ratio relative to Nov 9, 00:00 UTC; the display is capped at 10×. The notebook retains uncapped ratios for inspection.*

# Tech Stack
• Data Access: xarray + fsspec + s3fs (with zarr and h5netcdf) to stream NWM data directly from AWS S3 without downloading large archives.
• Spatial Indexing: h3 (v4+) and pandas. Hexagon polygons are built directly from H3 boundaries, so no geopandas or shapely is needed.
• Web Display: a hand-written MapLibre GL JS page (no Python mapping library) that animates 3D fill-extrusions.

# Step-By-Step Python Workflow

Step 1: Build a Spatial Hex-Lookup (reach coordinates → H3)

• Why: Operational NWM streamflow files only contain river reach IDs (feature_id), not coordinates.
• What the script does: It reads each reach's latitude and longitude from the NWM v3.0 retrospective Zarr store (`chrtout.zarr`), filters to the Pacific Northwest (42°N–49°N, -124.8°W to -116.5°W), assigns each reach to an H3 Resolution 6 hexagon (~36 km²), and caches the lookup as pnw_routelink.parquet so later runs are fast. (The RouteLink_CONUS.nc file is not hosted in this bucket, so the Zarr store is used for coordinates only.)

Step 2: Stream Recent Atmospheric River Data from AWS S3

• Why: NOAA publishes hourly NWM analysis_assim (assimilated streamflow) files on AWS Open Data (s3://noaa-nwm-pds), but only for a rolling window of about four weeks.
• What the script does: It streams 6-hourly snapshots for Nov 9–14, 2026 (24 frames) with fsspec and xarray, keeping only the PNW reaches in memory instead of downloading the full national files. Export stops with an explicit error if any expected snapshot is unavailable, rather than silently shifting the baseline or animation timeline.

Step 3: Select Reaches & Compute Baseline Ratios

• Why: Raw discharge varies greatly among rivers, so the map shows within-reach change rather than comparing absolute flow values.
• What the script does: Reaches missing any frame are excluded. Repeated >10× step changes are flagged for review but do not automatically disqualify a reach. Each cell selects the reach with the highest mean flow over the window. Cells whose selected reach starts at or below 1 m³/s are excluded; for every retained cell, each frame is divided by the actual Nov 9 flow, so the first frame is 1.0. The uncapped ratio is retained to three decimal places, while the map's display value is capped at 10×. Neither a large ratio nor a flagged jump establishes flood severity or bad data.

Step 4: Export GeoJSON & JS Data File

• Why: A static data file plus a single HTML page can be hosted on GitHub Pages with no server-side code.
• What the script does: It writes the hexagon polygons, capped display ratios, uncapped ratios, selected reach IDs, quality flags, and model/metric metadata to a minified pnw_hex_surge.geojson and pnw_hex_data.js (`window.HEX_DATA = ...`). map.html continues to animate the capped `surge` values; exposing the extra inspection fields in the map is a follow-up. The script-tag data load works on a server, GitHub Pages, or a double-click.

Step 5: GPU-Accelerated MapLibre Animation

• How the web map works: The whole capped display series is stored in each hexagon's `surge` array. Because MapLibre turns array properties into strings, the page first copies each frame's value into its own property (`s0`, `s1`, ...). Each frame, it updates fill-extrusion-height and fill-extrusion-color with the expression `["get", "s" + frameIndex]`, so the geometry is never reloaded. Height and color show the capped relative-flow ratio; the map still needs an in-frame explanation and reach-level inspection.


In [ ]:
import os
import json
import h3
import fsspec
import numpy as np
import xarray as xr
import pandas as pd

SURGE_CAP = 10.0          # display cap; uncapped ratios are retained in the export
JUMP_FACTOR = 10.0        # a step-to-step change larger than this (up or down) is a "jump"
MAX_JUMPS = 2             # repeated jumps trigger a review flag, not automatic exclusion

# ==========================================
# 1. FETCH & CACHE PNW COORDINATE LOOKUP
# ==========================================
ROUTELINK_CACHE = "pnw_routelink.parquet"

if os.path.exists(ROUTELINK_CACHE):
    meta = pd.read_parquet(ROUTELINK_CACHE)
else:
    print("Reading NWM reach coordinates from the retrospective Zarr (one-time)...")
    # RouteLink_CONUS.nc is not hosted in this bucket; the Zarr store carries
    # per-reach latitude/longitude keyed by feature_id.
    zarr_url = "s3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/chrtout.zarr"
    ds_rl = xr.open_zarr(zarr_url, storage_options={"anon": True})
    meta = pd.DataFrame({
        "feature_id": ds_rl["feature_id"].values,
        "lat": ds_rl["latitude"].values,
        "lon": ds_rl["longitude"].values,
    })

    # Filter to Pacific Northwest bounding box (OR, WA, parts of ID)
    meta = meta[
        (meta["lat"].between(42.0, 49.0)) &
        (meta["lon"].between(-124.8, -116.5))
    ].copy()

    # Assign H3 Resolution 6 hexagons (~36 km² cells)
    meta["h3_cell"] = [
        h3.latlng_to_cell(la, lo, 6) for la, lo in zip(meta["lat"], meta["lon"])
    ]
    meta.to_parquet(ROUTELINK_CACHE, index=False)

pnw_ids = set(meta["feature_id"])
print(f"Tracking {len(pnw_ids):,} PNW river reaches across {meta['h3_cell'].nunique():,} hexagons.")

# ==========================================
# 2. STREAM THIS WEEK'S NWM DATA (NOV 9–14, 2026)
# ==========================================
dates = pd.date_range("2026-11-09 00:00", "2026-11-14 18:00", freq="6h")
ts_dict = {}

for dt in dates:
    d_str = dt.strftime("%Y%m%d")
    h_str = dt.strftime("%H")
    url = f"s3://noaa-nwm-pds/nwm.{d_str}/analysis_assim/nwm.t{h_str}z.analysis_assim.channel_rt.tm00.conus.nc"

    try:
        with fsspec.open(url, anon=True) as f:
            ds = xr.open_dataset(f, engine="h5netcdf")
            mask = pd.Series(ds["feature_id"].values).isin(pnw_ids).values
            ids = ds["feature_id"].values[mask]
            flow = ds["streamflow"].values[mask]
            ts_dict[dt] = pd.Series(flow, index=ids)
    except Exception as e:
        print(f"Skipped {dt.strftime('%Y-%m-%d %H:%M UTC')}: {e}")
print(f"Loaded {len(ts_dict)} of {len(dates)} time steps.")
missing_dates = [dt for dt in dates if dt not in ts_dict]
if missing_dates:
    missing_text = ", ".join(dt.strftime("%Y-%m-%d %H:%M UTC") for dt in missing_dates)
    raise RuntimeError(f"Cannot export an evenly spaced animation; missing NWM snapshots: {missing_text}")

# ==========================================
# 3. SELECT REACHES & COMPUTE BASELINE RATIOS
# ==========================================
q_df = pd.DataFrame(ts_dict)
flow_cols = [c for c in q_df.columns if c != "h3_cell"]
complete = q_df[flow_cols].notna().all(axis=1)
dropped_incomplete = int((~complete).sum())
q_df = q_df.loc[complete].copy()
print(f"Skipping {dropped_incomplete:,} reaches with incomplete time series.")
if q_df.empty:
    raise RuntimeError("No reaches have a complete time series for all expected snapshots.")
q_df["h3_cell"] = meta.set_index("feature_id")["h3_cell"]

# Flag repeated large changes between 6-hourly steps for review. These can reflect
# regulation, model behavior, or real events; the flag is not a quality verdict.
# Flows are floored at 1 m³/s for jump detection so near-zero values do not dominate.
arr = q_df[flow_cols].clip(lower=1.0).to_numpy()
step = arr[:, 1:] / arr[:, :-1]
q_df["_jumps"] = ((step > JUMP_FACTOR) | (step < 1 / JUMP_FACTOR)).sum(axis=1)
q_df["_jump_flag"] = q_df["_jumps"] > MAX_JUMPS
print(f"Flagging {int(q_df['_jump_flag'].sum()):,} reaches with more than {MAX_JUMPS} jumps of {JUMP_FACTOR:g}x for review.")

# Use the highest-mean-flow reach in each hexagon as its representative series.
# The jump flag is retained rather than used as an automatic exclusion rule.
q_df["_mean"] = q_df[flow_cols].mean(axis=1)
main_reach = q_df.sort_values("_mean", ascending=False).groupby("h3_cell").head(1).copy()
main_reach["feature_id"] = main_reach.index
hex_ts = main_reach.set_index("h3_cell")[flow_cols]
reach_by_cell = main_reach.set_index("h3_cell")["feature_id"]
jump_flag_by_cell = main_reach.set_index("h3_cell")["_jump_flag"]

# Exclude cells whose representative reach starts at or below 1 m³/s.
baseline = hex_ts.iloc[:, 0]
eligible = baseline > 1.0
hex_ts = hex_ts.loc[eligible].copy()
reach_by_cell = reach_by_cell.loc[eligible]
jump_flag_by_cell = jump_flag_by_cell.loc[eligible]
if hex_ts.empty:
    raise RuntimeError("No representative reaches remain after applying the 1 m³/s baseline threshold.")
baseline = hex_ts.iloc[:, 0]
peak_flow = hex_ts.max(axis=1)

# Preserve the true per-reach ratio; only the animated display series is capped.
surge_raw = hex_ts.div(baseline, axis=0).round(3)
surge_display = surge_raw.clip(upper=SURGE_CAP).round(2)
timestamps = [t.strftime("%b %d, %H:%M UTC") for t in surge_display.columns]
print(f"Kept {len(hex_ts):,} hexagons.")

# ==========================================
# 4. EXPORT GEOJSON & JS DATA FILE
# ==========================================
features = []
for cell_id, row in surge_display.iterrows():
    raw_row = surge_raw.loc[cell_id]
    boundary = h3.cell_to_boundary(cell_id)
    coords = [[round(lon, 4), round(lat, 4)] for lat, lon in boundary]
    coords.append(coords[0])  # Close polygon ring

    features.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [coords]},
        "properties": {
            "h3": cell_id,
            "feature_id": int(reach_by_cell.loc[cell_id]),
            "jump_flag": bool(jump_flag_by_cell.loc[cell_id]),
            "base_cms": round(float(baseline.loc[cell_id]), 1),
            "peak_cms": round(float(peak_flow.loc[cell_id]), 1),
            "max_surge": round(float(row.max()), 2),
            "max_surge_raw": round(float(raw_row.max()), 3),
            "surge": row.values.tolist(),
            "surge_raw": raw_row.values.tolist(),
        }
    })

geojson_out = {
    "type": "FeatureCollection",
    "metadata": {
    "model": "NOAA National Water Model v3.0 analysis_assim",
        "baseline_utc": dates[0].strftime("%Y-%m-%dT%H:%MZ"),
        "h3_resolution": 6,
        "minimum_baseline_exclusive_cms": 1.0,
        "display_cap": SURGE_CAP,
        "metric": "streamflow divided by first-frame streamflow",
        "representative_reach": "highest mean flow in each H3 cell",
    },
    "timestamps": timestamps,
    "features": features
}
payload = json.dumps(geojson_out, separators=(",", ":"))

# Plain GeoJSON for reuse in other tools
with open("pnw_hex_surge.geojson", "w") as f:
    f.write(payload)
print(f"Saved pnw_hex_surge.geojson ({len(features)} hexagons, {len(timestamps)} frames).")

# Same data as a script that map.html loads with a <script> tag. Unlike fetch(),
# this works from a web server, GitHub Pages, and when opened from file://.
with open("pnw_hex_data.js", "w", encoding="utf-8") as f:
    f.write(f"window.HEX_DATA = {payload};\n")
print("Saved pnw_hex_data.js (loaded by map.html).")


# Caveats

This map shows **modeled** NOAA National Water Model v3.0 analysis flows, not observations. It describes change during one event; it is not a historical anomaly, flood extent, flood probability, or flood-severity map.

• **Metric and eligibility:** each retained reach is divided by its actual Nov 9, 00:00 UTC flow, so its first frame is 1.0. Reaches starting at or below 1 m³/s are excluded. The map caps displayed values at 10×, while the exported `surge_raw` series retains uncapped ratios. A large relative change does not imply high absolute discharge or danger.

• **Representative reach:** each cell shows one modeled reach (the highest-mean-flow reach assigned to that H3 cell), not all tributaries, a basin total, or conditions throughout the cell. The selected `feature_id` is retained in the export.

• **Jump flags:** repeated >10× step changes are flagged for review. This is a heuristic, not proof of bad data; flagged reaches are not automatically discarded.

• **Approximate gauge spot-check:** nearest NWM reaches and provisional USGS gauges for Nov 9–14 gave the following values. This small, approximate comparison is not an independent or basin-wide model validation:

| River | USGS, first → peak (m³/s) | NWM, first → peak (m³/s) |
|---|---|---|
| Chetco near Brookings | 3.8 → 85 | 13.5 → 386 |
| Sixes at Sixes | 2.0 → 32 | 3.5 → 159 |
| South Fork Coquille at Powers | 0.5 → 19 | 1.2 → 42 |

These station comparisons are approximate because they use nearest reaches and 6-hourly model snapshots. Check an observed source such as the USGS National Water Dashboard for current station conditions.

• **Completeness:** all 24 expected snapshots are required. If a file is missing or cannot be read, the workflow reports the missing times and stops rather than exporting a misleading equally spaced animation.


Part 2: Animated MapLibre HTML Map (map.html)

Instead of reloading geometries on every frame, the HTML page keeps each capped display series in the `surge` property (flattened to `s0`, `s1`, ... on load) and updates MapLibre's fill-extrusion-height and fill-extrusion-color expressions with `["get", "s" + frameIndex]`. It includes a play/pause button, a time slider, and a legend. The new reach IDs, jump flags, and uncapped ratios are exported but are not yet exposed by the map. Add a 2D option, reach-level inspection, and a clearer mobile-visible model/metric explanation in `map.html` as follow-up work. The data comes from `pnw_hex_data.js`, loaded with a script tag rather than `fetch`, so the page works on a web server, on GitHub Pages, and when opened directly from disk (it still needs an internet connection for MapLibre and the basemap).

The collapsed section below shows the current map.html from this folder, so it always matches the file on disk.


In [16]:
#| echo: false
import html
from IPython.display import HTML

with open("map.html", "r", encoding="utf-8") as f:
    map_src = f.read()

HTML(
    "<details><summary>Show the map.html source</summary>"
    f"<pre><code>{html.escape(map_src)}</code></pre></details>"
)